# reproduce_flow

Reproduces the results of the thesis report ([thesis_final_report.pdf](thesis_final_report.pdf), repo root) for one fixed model, end to end, along the **maximum-FPS flow**:

**dataset → trained model → ILP solver under constraints → QAT → deploy with FINN → verify with the bare-metal app**

Cost-model calibration, the platform comparison and the limitations have their own sections. Section, table and figure numbers (Sec. V-B, Table IV, ...) refer to the report. Where a stage has a result in the report, the expected value is stated next to it.

## Handover checklist (to be filled)

Tick an item when it is done and checked, and write the date and any result next to it. Nothing is ticked yet: the status line at the end of each section says what was and was not run when the notebook was written, and several items below are also a `TODO` in the section itself.

Handed over by: ______________________ Date: ____________ Received by: ______________________ Date: ____________

**General**
- [ ] The notebook and the rest of the `philips_handover` branch are committed and pushed
- [ ] `thesis_final_report.pdf` is on the branch (it is git-ignored by `*.pdf`, so it needs `git add -f`)
- [ ] The stored artifacts of 1.2 are on the branch or delivered another way. Checkpoints (`*.pth`) are git-ignored; the `.pth.txt` copies are not. Delivery route: ______________________
- [ ] The folders that are out of scope (1.1) are removed or marked, as agreed
- [ ] Stale paths are fixed (`report/MILP/...` in `run_trt_int8.sh` and docstrings, `ILP/artifacts/S12_dense_bilinear_512_uniform_v1/`)

**2. Dataset preparation**
- [ ] The Kaggle download has the folder layout the builder reads (`archive/dataset-prep/ARCADE/data/syntax/{train,val,test}`)
- [ ] `prepare_arcade_multiclass.py` was run and gave 1200 training and 300 test cases
- [ ] The way the training split is delivered is decided (only the test split is tracked in git)

**3. Trained model**
- [ ] The network check gives 24,261 (Ours-S) and 361,452 (Ours-L) parameters
- [ ] The FP32 checkpoint `12_dense_relu_warmstart150ep` is delivered
- [ ] The rows in `nas/results.csv` match Table IV (Dice 0.7876, 0.7954, 0.8279)
- [ ] Optional: the training was re-run on Snellius. Result: ______________________

**4. Cost-model calibration**
- [ ] `build_error_summary.py` reproduces Table V
- [ ] The stored synthesis data of the four constrained builds is on the branch (`report/hw/constrained_builds/`, `dataflow/builds/S12_512_ablation_v2/`)
- [ ] Optional: the refit was run without `--apply`. Result: ______________________

**5. ILP solver, max-FPS mode**
- [ ] The Hessian sensitivity file is delivered, and the `layer_sensitivity.py` flags (1200 images, 40 probes, seed 0) are confirmed
- [ ] `run_maxfps.sh` was re-run, or the stored `maxfps/` and `pinned_minres/` outputs are confirmed
- [ ] The `net_explicit.py` flags are confirmed and `simfifo_pairs/layer_bits_folding_S12_512_bilin_int8_maxfps_pairs.json` is delivered

**6. QAT**
- [ ] The INT8 QAT checkpoint is delivered
- [ ] A Slurm job for the INT8 QAT run is added to `nas/slurm/` (only the INT6 job is stored)
- [ ] The uniform INT8 bit file has a permanent location (now under `ILP/artifacts/archive/`)
- [ ] The Dice difference is explained: report 0.7946, stored 0.7930 (software) and 0.7943 (board)

**7. Deploy with FINN**
- [ ] The FINN version is recorded: v0.10.1 plus the cherry-picked MVAU fix (commit hash, and any other uncommitted change, or a patch file)
- [ ] The FINN container (image or recipe) is delivered or described
- [ ] The build outputs are copied from the build PC: QONNX export, per-partition results, post-implementation utilization report
- [ ] The launch scripts are checked (the `PDIR` edit, the three-build assembly, the list of helper scripts copied into the container)
- [ ] The Vivado base project and `s12_512_bilin_int8_v2.xsa` (and the bitstream) are delivered
- [ ] Optional: the FINN stage was re-run on the build PC. Result: ______________________

**8. Verify with the bare-metal app**
- [ ] The bare-metal sources (`lwip_collect_results`, `lwip_stream`) are copied from the build PC into the repo
- [ ] Build and programming steps for the Vitis project and the board network setup are written down
- [ ] `collect_results.py --power` was run on the board. Result (report: 354.8 FPS, 6.87 ms, 16.1 and 5.2 mJ): ______________________

**9. Platform comparison**
- [ ] The TensorRT installation (`INSTALL_TRT.md`) works
- [ ] The CPU and GPU benchmarks were re-run and the numbers recorded (they vary from run to run): ______________________
- [ ] The DPU flow was run on the ZCU104, including the QAT model
- [ ] Table III is compared with the new numbers, and the differences are accepted

**10. Limitations**
- [ ] The author has reviewed the list and added what is missing

**Appendix A. Max-accuracy flow**
- [ ] Decision: keep the placeholder, finish steps A.3 to A.5, or remove it: ______________________

## 1. Overview

### 1.1 Scope

One run of this notebook takes **one model** through the flow: **Ours-S** (default; the model deployed on the FPGA in the report) or **Ours-L** (the full-width variant), chosen in the Setup cell (1.3). The architecture is fixed. The flow is the **maximum-FPS flow** of Sec. IV-B4: uniform 8-bit quantization, resource caps at 100%, folding chosen to maximize throughput.

**In scope.** Reproducing these results:

| Section | Stage | Report | Result to reproduce |
|---|---|---|---|
| 2 | Dataset preparation | Sec. II-A | Dataset509: 1200 train/validation and 300 test frames, 512 × 512, 4 classes |
| 3 | Trained model (FP32) | Sec. II-B, V-A, Tables I and IV | Ours-S Dice 0.7876, and 0.7954 after continued training. Ours-L Dice 0.8279 |
| 4 | Cost-model calibration | Sec. IV-B6, V-B1, Table V | Mean absolute error of the calibrated model: LUT 13.6%, BRAM 5.9%, DSP 9.4% |
| 5 | ILP solver, max-FPS mode | Sec. IV-B3, IV-B4, IV-B7 | Folding and FIFO depths for uniform 8 bit, caps 100% |
| 6 | QAT | Sec. IV-B8 | Ours-S, 8 bit: Dice 0.7946 after QAT (see the note in Section 6) |
| 7 | Deploy with FINN | Sec. V-B2 | 157k LUT, 156 BRAM, 449 DSP at 187.48 MHz |
| 8 | Verify with the bare-metal app | Sec. V-C, Fig. 6, Table III (FPGA row) | 354.8 FPS, 6.87 ms latency, 16.1 mJ per image |
| 9 | Platform comparison | Sec. V-C, Table III | FPGA vs GPU, CPU and DPU: latency, throughput, energy per image |
| 10 | Limitations | Sec. V-C, VI | |

**Optional, off by default.** Appendix A is a placeholder for the maximum-accuracy flow of Sec. IV-B5 and Fig. 5 (150 FPS, caps at 40%, bit widths 4/6/8 per layer). It is switched on with `RUN_MAX_ACCURACY_FLOW` in the Setup cell.

**Out of scope.** The investigations behind the design, although they appear in the report:

- the staged greedy ablation and Table II (activation, upsampling, width and context-stage variants), including the max-unpooling study (Fig. 4)
- the benchmark of other models (other rows of Table IV)
- ReLU vs PReLU (Sec. V-A, `report/hw/activation/`)
- the maximum-accuracy flow beyond the placeholder of Appendix A. Table V reuses the stored results of those builds (Section 4)
- the comparison with FINN's own solver and FIFO sizing (Sec. V-B3)

**Ours-L.** The repo has no ILP configuration, QAT bit file or FINN build scripts for it, and the report has no hardware results for it (only its FP32 Dice, Table IV). For Ours-L only Sections 2 and 3 apply.

### 1.2 Reusing existing artifacts

Every long or unavailable step has a `REUSE_*` flag in the Setup cell (1.3). With a flag set to `True` (the default) the stage is skipped, the stored artifact is checked for and printed, and the flow continues from it. Set a flag to `False` to run the stage.

| Flag | Section | Stored artifact |
|---|---|---|
| `REUSE_DATASET` | 2 | `data/nnUNet_raw/Dataset509_ARCADE_1x1_4c/` |
| `REUSE_FP32_CHECKPOINT` | 3 | `data/nnUNet_results/Dataset509_ARCADE_1x1_4c/nnUNetTrainerENet_12_dense_relu_warmstart150ep__nnUNetPlans__2d/fold_0/checkpoint_best.pth` |
| `REUSE_COST_MODEL` | 4 | the constants in `ILP/finn_cost_model.py` and the stored post-synthesis results in `report/hw/constrained_builds/` |
| `REUSE_HESSIAN` | 5 | `ILP/artifacts/layer_sensitivity_12_dense_relu_warmstart150ep_n1200p40_seed0.json` |
| `REUSE_ILP_SOLUTION` | 5 | `ILP/artifacts/S12_512_bilin_int8_maxfps/maxfps/` and `pinned_minres/` |
| `REUSE_FIFO_TUNING` | 5 | `ILP/artifacts/S12_512_bilin_int8_maxfps/simfifo_pairs/layer_bits_folding_S12_512_bilin_int8_maxfps_pairs.json` |
| `REUSE_QAT_CHECKPOINT` | 6 | `data/nnUNet_results/Dataset509_ARCADE_1x1_4c/nnUNetTrainerLayerQuantEnetFINN_12_dense_relu_bilinear_upsample_256_perlayer_12_dense_relu_warmstart150ep_bilinear_uniform_int8_perlayer_ft15ep__nnUNetPlans__2d/fold_0/checkpoint_best.pth` |
| `REUSE_QONNX_EXPORT` | 7 | `dataflow/builds/S12_dense_512_bilinear_int8_maxfps_v1/outputs/` |
| `REUSE_FINN_BUILD` | 7 | `dataflow/builds/S12_dense_512_bilinear_int8_maxfps_v1/results/` and `deployment/xsa/` |
| `REUSE_BOARD_RESULTS` | 8 | `deployment/results.csv` and `deployment/power_results.csv` |
| `REUSE_BENCHMARKS` | 9 | `report/hw/final_results/benchmark_s12_512.csv` |

A checkpoint may be stored as `checkpoint_best.pth` or as `checkpoint_best.pth.txt`, a copy under a name that git does not ignore (`*.pth` is git-ignored). The cells accept both.

All stored artifacts belong to Ours-S. For Ours-L set the flags of Sections 2 and 3 to `False`.

### 1.3 Setup

Run this cell first. It selects the model, sets the reuse flags, and defines the helpers the other cells use. Commands for the training environment go through `docker exec` into `TRAIN_CONTAINER`; set it to `None` if this notebook already runs inside the training environment.

In [ ]:
import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()                      # this notebook lives in the repo root
MODEL = 'ours-s'                            # 'ours-s' (default) or 'ours-l'

TRAIN_CONTAINER = 'lightmunet_dev'          # training environment (see 1.5); None = already inside it
CONTAINER_REPO = '/workspace/LightM-UNet'   # the repo as mounted inside the containers
FINN_CONTAINER = 'finn_persistent'          # the name changes when the FINN container is recreated: check docker ps

# Reuse flags: True = use the artifact that is already in the repo and skip the stage (see 1.2)
REUSE_DATASET = True
REUSE_FP32_CHECKPOINT = True
REUSE_COST_MODEL = True
REUSE_HESSIAN = True
REUSE_ILP_SOLUTION = True
REUSE_FIFO_TUNING = True
REUSE_QAT_CHECKPOINT = True
REUSE_QONNX_EXPORT = True
REUSE_FINN_BUILD = True                     # the FINN container is on the build PC: keep True anywhere else
REUSE_BOARD_RESULTS = True
REUSE_BENCHMARKS = True

# Appendix A: the max-accuracy flow (150 FPS, 40% caps, bits 4/6/8). A placeholder, off by default.
RUN_MAX_ACCURACY_FLOW = False

DATASET_ID = 509
DATASET_NAME = 'Dataset509_ARCADE_1x1_4c'
TRAINER_CLASS = 'nnUNetTrainerENet'
RESULTS = REPO_ROOT / 'data' / 'nnUNet_results' / DATASET_NAME


def inside(path):
    # a repo path as the training environment sees it
    p = Path(path)
    if TRAIN_CONTAINER:
        return CONTAINER_REPO + '/' + p.relative_to(REPO_ROOT).as_posix()
    return str(p)


def sh(cmd, container=None, env=None, check=True):
    # run cmd (a list) from the repo root, here or through docker exec in a container
    env = env or {}
    if container:
        full = ['docker', 'exec', '-w', CONTAINER_REPO]
        for k, v in env.items():
            full += ['-e', k + '=' + v]
        return subprocess.run(full + [container] + list(cmd), check=check)
    merged = dict(os.environ)
    merged.update(env)
    return subprocess.run(list(cmd), cwd=str(REPO_ROOT), env=merged, check=check)


def existing(path):
    # the file, or its .txt copy (checkpoint_best.pth.txt), or None
    p = Path(path)
    for q in (p, p.with_name(p.name + '.txt')):
        if q.exists():
            return q
    return None


def require(path, flag):
    found = existing(path)
    if found is None:
        raise FileNotFoundError('{} is missing: set {} = False to produce it'.format(path, flag))
    print('reusing', found.relative_to(REPO_ROOT).as_posix())
    return found


def read_csv(path):
    with open(str(path), newline='') as f:
        return list(csv.DictReader(f))


print('model:', MODEL, '| repo:', REPO_ROOT)

### 1.4 Verification status

The FINN build container (with Vivado and Vitis HLS 2022.2), its output artifacts and the bare-metal application sources are on the build PC, not on the machine this notebook was prepared on. The FINN stage and the board stage were therefore not re-run for this handover: their cells call the existing scripts in the repo as they are, and the stored results are reused. To run them, work on the build PC and set the reuse flags to `False`. Sections 2 to 9 each end with a status line saying what has and has not been checked.

### 1.5 Environments

The stages run in different, non-interchangeable environments.

| Stage | Environment |
|---|---|
| 2, 3, 4, 5, 6, and CPU/GPU in 9 | **Training environment:** Docker image `pytorch/pytorch:2.0.1-cuda11.7-cudnn8-devel`, container `lightmunet_dev`, repo mounted at `/workspace/LightM-UNet`. **Needs an NVIDIA GPU with CUDA** (see below). Started and set up as shown below. |
| 3, 6, and DPU QAT in 9 | **Snellius HPC cluster (SURF) with Slurm:** the report's FP32 and QAT training ran there as batch jobs (`nas/slurm/*.job`, `dpu/slurm/*.job`), on one `gpu_mig` GPU per job, with a conda environment instead of the Docker image. See 3.5 |
| 7 Deploy with FINN | **FINN container** on the build PC (name `finn_persistent`, changes when recreated: `docker ps`), with FINN **v0.10.1 plus a cherry-picked MVAU fix from `v1.0.0-alpha`** (see 10.3), QONNX and Brevitas installed editable under `/tmp/home_dir/.local`. Run scripts as `docker exec -e HOME=/tmp/home_dir <container> python3 <script>`. Vivado and Vitis HLS 2022.2 are at `/tools/Xilinx/` and need `source .../settings64.sh` first |
| 8 Bare-metal verify | ZCU106 board running the bare-metal application (sources on the build PC), and a PC on the same network running `deployment/collect_results.py` |
| DPU in 9 | Vitis AI image `xilinx/vitis-ai-pytorch-cpu:ubuntu2004-3.0.0.106` in WSL, and a ZCU104 board (`dpu/README.md`) |

**Starting the training container.** Replace `<REPO_PATH>` with the absolute path of this repo on the host, for example `C:\Users\<user>\...\LightM-UNet` on Windows or `/home/<user>/.../LightM-UNet` on Linux. Keep the quotes around the `-v` argument if the path contains spaces.

```
docker run --rm -it --gpus all --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --name lightmunet_dev -v "<REPO_PATH>:/workspace/LightM-UNet" -w /workspace/LightM-UNet/enet pytorch/pytorch:2.0.1-cuda11.7-cudnn8-devel
```

`--name lightmunet_dev` is added to the command that was used originally, so that the notebook can reach the container with `docker exec lightmunet_dev ...`. The container is interactive and removed when you exit (`--rm -it`): keep that terminal open while the notebook runs, or set `TRAIN_CONTAINER = None` in the Setup cell and run the notebook's commands from inside the container. The shell starts in `enet/`. Set the environment up once per container, from the repo root (the script installs Brevitas 0.12.1, QONNX, pulp and the nnU-Net dependencies, and runs `pip install -e .` in `enet/`):

```
cd /workspace/LightM-UNet && bash scripts/setup-enet.sh
```

**This assumes an NVIDIA GPU with CUDA.** `--gpus all` needs an NVIDIA GPU, a current driver and the NVIDIA Container Toolkit (on Windows, Docker Desktop with the WSL2 backend and GPU support). `scripts/setup-enet.sh` stops unless it finds torch 2.0.1, CUDA 11.7 and a visible GPU, and the training, Hessian, QAT and benchmark steps all use the GPU. A machine without one can only reuse the stored artifacts (the reuse flags in 1.2).

Target device: AMD XCZU7EV (`xczu7ev-ffvc1156-2-e`), on the ZCU106 board.

### 1.6 Directory tree

Numbers in brackets are the notebook sections that use a folder.

```
LightM-UNet/
├── reproduce_flow.ipynb          this notebook
├── thesis_final_report.pdf       the report every stage is checked against
├── AGENTS.md                     repo conventions, the environments, known sharp edges
│
├── enet/                         [3,6] nnU-Net v2 fork: models and trainers (training environment)
│   └── nnunetv2/
│       ├── nets/                     ENet.py (FP32 network), LayerQuantEnetFINN.py (quantized network that is exported to FINN)
│       ├── training/nnUNetTrainer/   nnUNetTrainerENet.py (FP32 trainer, configured by ENET_* env vars),
│       │                             nnUNetTrainerLayerQuantEnetFINN_*.py (QAT trainers)
│       └── dataset_conversion/       nnU-Net dataset converters
├── data/                         [2,3,6] nnU-Net data roots
│   ├── nnUNet_raw/Dataset509_ARCADE_1x1_4c/   512x512 ARCADE, 4 classes (LAD, RCA, LCX, LM); only the test split is tracked in git
│   ├── nnUNet_preprocessed/
│   └── nnUNet_results/           checkpoints and logs, one folder per run
├── nas/                          [3,6] training jobs, calibration, results table
│   ├── slurm/                        stage_12_dense_relu*.job: the FP32 training jobs; qat_*.job: QAT jobs
│   ├── post-quantization/            calibrate_*.py: quantizer-scale calibration before QAT
│   ├── collect_results.py            evaluates a run and upserts one row into results.csv
│   └── results.csv                   one row per trained model (parameters, FLOPs, Dice per class)
│
├── ILP/                          [4,5] resource-aware ILP solver
│   ├── layer_sensitivity.py          Hessian sensitivity per layer and bit width (HAWQ)
│   ├── finn_milp.py                  the solver: bit widths and folding under LUT/BRAM/DSP caps (docs: finn_milp.md)
│   ├── finn_cost_model.py            calibrated LUT/BRAM/DSP/cycle model (docs: finn_cost_model.md)
│   ├── calibration.csv               log of every fitted cost-model constant
│   ├── fifo_model.py                 analytical FIFO sizing
│   ├── expand_layer_bits.py          per-layer bits to per-site bits, for QAT and export
│   ├── analytical/                   dataflow-graph model and FIFO tuning simulation (net_explicit.py)
│   ├── configs/                      config_*.py: per-model solver configuration, selected with --config
│   └── artifacts/                    solver outputs, one folder per build (S12_512_bilin_int8_maxfps/ is the final build)
│
├── dataflow/                     [4,7] FINN hardware build (runs in the FINN container on the build PC)
│   ├── s12_export.py                 checkpoint + bits to QONNX export and conv-order dump
│   ├── finn_s12_preamble.py          FINN preamble: graph transforms before the build
│   ├── finn_s12_build.py             partitioned FINN build (with finn_s12_build_steps.py, finn_stage_partition.py)
│   ├── builds/                       one folder per build job: export and launch scripts, outputs, results
│   ├── utils/                        fit_cost_model_from_node_csv.py (cost-model refit) and other build helpers
│   ├── datasets/                     node-level post-route data used to calibrate the cost model
│   ├── collect_results.py            gathers build resource results into results.csv
│   └── README.md
├── deployment/                   [8]   board side
│   ├── xsa/                          hardware platform files (.xsa) and bitstreams; the report's design is s12_512_bilin_int8_v2.xsa
│   ├── drivers/                      IODMA driver sources
│   ├── img_intf/                     PC to board image transfer client (Python) and protocol tests
│   ├── collect_results.py            test-split evaluation on the board, writes results.csv (--power for energy)
│   └── FINN_interface/               Vitis IDE workspace metadata only; the bare-metal sources are on the build PC, not in the repo
│
├── report/                       thesis scripts, tables and figures
│   ├── sw/                           accuracy side: ablation, benchmarks, max-unpool and dilation studies, final-model plots
│   └── hw/                           hardware side
│       ├── constrained_builds/   [4]   model-vs-synthesis error (Table V), synthesis utilization of the constrained builds
│       ├── benchmark/            [9]   CPU and GPU benchmark scripts (ONNX Runtime, TensorRT, power and clock sampling)
│       └── final_results/        [8,9] benchmark_s12_512.csv (all platforms) and the final INT8 results
├── dpu/                          [9]   Vitis AI DPU baseline on the ZCU104 (flow, QAT, benchmark)
├── scripts/                      setup-enet.sh (builds the training environment), checkpoint_sync.sh (rsync checkpoints from the HPC)
├── git-checkpoint-sync           wrapper so that `git checkpoint-sync` runs scripts/checkpoint_sync.sh
├── sources/                      FINN checkout (reference only) and the ENet paper
└── archive/                      retired code and docs, kept for history
    └── dataset-prep/             [2]   ARCADE to nnU-Net dataset builders (prepare_arcade_multiclass.py)
```

## 2. Dataset preparation

The report uses the ARCADE dataset (Popov et al., Scientific Data 2024, report ref. [5]), its SYNTAX subset: coronary angiography frames of 512 × 512 pixels, annotated with polygons for the 26 artery segments of the SYNTAX score. After the split there are 1000 training, 200 validation and 300 test frames. The 26 segments are grouped into the four main branches (Sec. II-A), which gives the nnU-Net dataset `Dataset509_ARCADE_1x1_4c` with the labels 0 background, 1 LAD, 2 RCA, 3 LCX and 4 LM.

### 2.1 Download

Download the dataset from Kaggle: [https://www.kaggle.com/datasets/nirmalgaud/arcade-dataset](https://www.kaggle.com/datasets/nirmalgaud/arcade-dataset)

Either use the website, or the Kaggle CLI (needs an API token in `~/.kaggle/kaggle.json`):

```
kaggle datasets download -d nirmalgaud/arcade-dataset
```

Unpack it so that this layout exists. It is the layout of the original ARCADE release, and the one the builder script reads:

```
archive/dataset-prep/ARCADE/data/
└── syntax/
    ├── train/   images/   annotations/train.json
    ├── val/     images/   annotations/val.json
    └── test/    images/   annotations/test.json
```

The `stenosis` subset is not used. The Kaggle page could not be opened to check that its folders match this layout. If they differ, move the folders so that the paths above exist: the cell in 2.2 lists what is missing. New files under `archive/` are git-ignored, so the downloaded data is not committed.

### 2.2 Build the 4-class nnU-Net dataset

`archive/dataset-prep/prepare_arcade_multiclass.py` turns the COCO polygon annotations into nnU-Net cases:

- Each polygon is rasterized with the class of its segment. The default grouping maps the COCO category ids 6 to 12 to LAD (1), ids 1, 2, 3, 4 and 20 to 23 to RCA (2), ids 13 to 19, 24 and 25 to LCX (3), and id 5 to LM (4). These are the ids of the annotation file, not the segment names. Categories outside the groups (the stenosis category) are dropped.
- Frames are converted to grayscale. `--grid 1x1` keeps each frame whole, with no tiling, so every case is a 512 × 512 image.
- Training and validation frames go to `imagesTr` and `labelsTr` (1200 cases), test frames to `imagesTs` and `labelsTs` (300 cases). Case ids look like `train_1000_p0000`, with `_0000.png` appended for images.
- It writes `dataset.json` and a `splits_final.json` with the 1000 training and 200 validation cases.

The script writes into `data/nnUNet_raw/Dataset509_ARCADE_1x1_4c/`, replacing what is there. `REUSE_DATASET = True` skips the build and only checks the existing dataset.

In [ ]:
ARCADE_DIR = REPO_ROOT / 'archive' / 'dataset-prep' / 'ARCADE' / 'data'
RAW = REPO_ROOT / 'data' / 'nnUNet_raw' / DATASET_NAME

if REUSE_DATASET:
    print('reusing', RAW.relative_to(REPO_ROOT).as_posix())
else:
    expected = []
    for split in ('train', 'val', 'test'):
        expected.append(ARCADE_DIR / 'syntax' / split / 'images')
        expected.append(ARCADE_DIR / 'syntax' / split / 'annotations' / (split + '.json'))
    missing = [p for p in expected if not p.exists()]
    if missing:
        raise FileNotFoundError('unpack the ARCADE download so that these exist:\n' + '\n'.join(str(p) for p in missing))
    sh(['python3', 'archive/dataset-prep/prepare_arcade_multiclass.py',
        '--dataset-id', str(DATASET_ID), '--source', 'syntax', '--grid', '1x1'], container=TRAIN_CONTAINER)

n_train = len(list((RAW / 'imagesTr').glob('*.png')))
n_test = len(list((RAW / 'imagesTs').glob('*.png')))
with open(str(RAW / 'dataset.json')) as f:
    labels = json.load(f)['labels']
with open(str(RAW / 'splits_final.json')) as f:
    split = json.load(f)[0]
print('imagesTr {} (expected 1200), imagesTs {} (expected 300)'.format(n_train, n_test))
print('split: {} train / {} val (expected 1000 / 200)'.format(len(split['train']), len(split['val'])))
print('labels:', labels)

**Status, Section 2.** The builder command and its defaults were read from `prepare_arcade_multiclass.py`; the case counts come from the existing `Dataset509_ARCADE_1x1_4c` (1200 and 300 cases). The download and the build were not run, and the Kaggle layout was not checked.

## 3. Trained model

This section produces the FP32 checkpoint that the rest of the flow starts from.

In this notebook the models are called **Ours-S** and **Ours-L**, as in the report. Names such as `12_dense_relu` below are configuration names only: they identify a run in folder names, scripts and `nas/results.csv`.

Both models share the training setup of Sec. II-B: ARCADE 512 × 512, 1000 training and 200 validation images, 150 epochs of 250 iterations, AdamW (learning rate 1e-3, weight decay 1e-2) with polynomial decay, cross-entropy plus soft Dice loss, nnU-Net v2 with its standard 2D augmentation. The checkpoint with the best validation Dice is kept.

### 3.1 Ours-S

The smallest model in the report, and the one deployed on the FPGA. It is ENet reduced to a quarter of its channel widths, with three further changes (Sec. V-A, Table I):

- **Width ×1/4.** Channels per stage 4, 16, 32, 16, 4, instead of ENet's 16, 64, 128, 64, 16.
- **Bilinear upsampling** in the decoder, instead of max-unpooling.
- **Dilation-only context stage.** Stage 2 has 16 dilated bottlenecks, with dilations 2, 4, 8, 16 repeated. It has no regular or asymmetric bottlenecks.
- **ReLU** instead of PReLU, because ReLU maps to a single FINN threshold unit.

Input: one 512 × 512 grayscale frame. Output: per-pixel scores for 5 classes (background, LAD, RCA, LCX, LM).

| | Ours-S |
|---|---|
| Parameters | 24,261 |
| FLOPs per image | 297.3 M |
| Mean Dice, first 150-epoch run (Table IV) | 0.7876 (LAD 0.6523, LCX 0.8090, RCA 0.9087, LM 0.7804) |
| Mean Dice after continued training (Sec. V-B2) | 0.7954 |

The deployed model is trained in two runs. The second run continues from the best checkpoint of the first for another 150 epochs at a lower learning rate (1e-4), through nnU-Net's `-pretrained_weights`. This takes the FP32 Dice from 0.7876 to 0.7954, and the second checkpoint is the starting point for the ILP, QAT and FINN stages.

### 3.2 Ours-L

The full-width variant of the same design (Table IV): the same bilinear decoder, dilation-only context stage and ReLU, with ENet's channel widths 16, 64, 128, 64, 16. It is trained in a single 150-epoch run.

| | Ours-L |
|---|---|
| Parameters | 361,452 |
| FLOPs per image | 4214.1 M |
| Mean Dice (Table IV) | 0.8279 (LAD 0.7038, LCX 0.8520, RCA 0.9166, LM 0.8391) |

### 3.3 Configuration

| | Ours-S | Ours-L |
|---|---|---|
| Training runs (config names) | `12_dense_relu`, then `12_dense_relu_warmstart150ep` | `12_dense_relu_u1` |
| `ENET_CHANNELS` | `4,16,32,16,4` | `16,64,128,128,64,16` |
| Slurm jobs in `nas/slurm/` | `stage_12_dense_relu.job`, then `stage_12_dense_relu_warmstart150ep.job` | `stage_12_dense_relu_ablation_array.job`, array task 2 (`sbatch --array=2 ...`) |
| FP32 checkpoint used by the rest of the flow | `12_dense_relu_warmstart150ep` | none: the flow stops here |

The 5-value and 6-value channel forms describe the same network (stage 3 repeats the stage 2 width in the 5-value form). The values above are copied from the jobs that produced the report.

In [ ]:
EPOCHS = 150

# Architecture settings shared by Ours-S and Ours-L (ENET_* env vars, read by nnUNetTrainerENet)
ARCH_ENV = {
    'ENET_BOTTLENECKS': '4,8,8,2,1',
    'ENET_CONTEXT_PATTERN': 'dense_dilation',   # dilation-only context stage
    'ENET_DECODER_TYPE': 'upsample_conv',       # bilinear upsampling
    'ENET_USE_PRELU': '0',                      # ReLU
    'ENET_USE_DILATED': '1',
    'ENET_USE_ASYMMETRIC': '0',
    'ENET_USE_STRIDED': '1',
    'ENET_USE_DSC': '0',
    'ENET_SEPARABLE_DILATED': '0',
    'ENET_DSC_NO_PROJECTION': '0',
}

# config_name identifies a run in folder names and in nas/results.csv.
# stage is the label nas/collect_results.py gets for that run (as in the Slurm jobs).
# dice is the value in the report: Table IV for the first Ours-S run and for Ours-L, Sec. V-B2 for the second Ours-S run.
MODELS = {
    'ours-s': {
        'channels': '4,16,32,16,4',
        'params': 24261,
        'flops_m': 297.3,
        'runs': [
            {'config_name': '12_dense_relu', 'stage': '12_dense_relu',
             'lr': '1e-3', 'warm_start_from': None, 'dice': 0.7876},
            {'config_name': '12_dense_relu_warmstart150ep', 'stage': '12_dense_relu_warmstart150ep',
             'lr': '1e-4', 'warm_start_from': '12_dense_relu', 'dice': 0.7954},
        ],
    },
    'ours-l': {
        'channels': '16,64,128,128,64,16',
        'params': 361452,
        'flops_m': 4214.1,
        'runs': [
            {'config_name': '12_dense_relu_u1', 'stage': '12_dense_relu_ablation',
             'lr': '1e-3', 'warm_start_from': None, 'dice': 0.8279},
        ],
    },
}
model = MODELS[MODEL]
print(MODEL, '->', [r['config_name'] for r in model['runs']])

### 3.4 Check the network

Builds the FP32 network for the selected model and counts its parameters. Run this where torch and the `enet` package are installed (the training environment). Expected: 24,261 for Ours-S and 361,452 for Ours-L.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / 'enet'))
from nnunetv2.nets.ENet import ENet

channels = tuple(int(c) for c in model['channels'].split(','))
net = ENet(in_channels=1, out_channels=5, channels=channels, bottlenecks_per_stage=(4, 8, 8, 2, 1),
           decoder_type='upsample_conv', use_dilated=True, use_asymmetric=False, use_strided=True,
           use_dsc=False, context_pattern='dense_dilation', use_prelu=False)
n_params = sum(p.numel() for p in net.parameters())
print('{}: {:,} parameters (report: {:,})'.format(MODEL, n_params, model['params']))

### 3.5 Train, or reuse the checkpoint

**Where the report's models were trained.** On the **Snellius** HPC cluster (SURF), as **Slurm** jobs: `nas/slurm/*.job`. Each job asks for one GPU on the `gpu_mig` partition (`--partition=gpu_mig --gpus=1 --cpus-per-task=8 --mem=32G`, with a 4 h limit for FP32 training and 2 h for QAT), loads the `2023` software environment and `CUDA/12.1.1`, activates the conda environment `~/conda-envs/lightmunet`, and expects the repo at `~/LightM-UNet`. Submit from the repo root with `sbatch nas/slurm/<job>.job`; logs go to `logs/`. The jobs are written for Snellius: on another cluster the partition, modules, conda environment and paths have to be changed, and the repo does not record how the conda environment was created. The cell below is the Docker equivalent of the same steps.

Finished checkpoints are copied back with `scripts/checkpoint_sync.sh` (`git checkpoint-sync` if the repo root is on your `PATH`), which rsyncs `data/nnUNet_results/` from the Snellius login node `snellius.surf.nl`. The user name, host, repo path and ssh key are set with the environment variables `HPC_USER`, `HPC_HOST`, `HPC_ROOT` and `SSH_KEY`. Compute-node host names only resolve from inside a running job.

With `REUSE_FP32_CHECKPOINT = True` the next cell only checks that the stored best checkpoint of every run exists. With `False` it runs the same steps as the Slurm jobs listed in 3.3, one run after the other: nnU-Net preprocessing (skipped if the plans exist), `nnUNetv2_train` on fold 0 (resumed with `--c` if a run was interrupted, skipped if it finished), then `nas/collect_results.py`, which evaluates the best checkpoint and writes the run's row into `nas/results.csv`. For the second Ours-S run, the best checkpoint of the first run is passed as `-pretrained_weights`.

Each run is 150 epochs, which takes hours on one GPU. The checkpoints land in `data/nnUNet_results/Dataset509_ARCADE_1x1_4c/nnUNetTrainerENet_<config name>__nnUNetPlans__2d/fold_0/checkpoint_best.pth`.

In [ ]:
COLLECT_FLAGS = {       # nas/collect_results.py flag -> ARCH_ENV variable
    '--decoder-type': 'ENET_DECODER_TYPE',
    '--use-dilated': 'ENET_USE_DILATED',
    '--use-asymmetric': 'ENET_USE_ASYMMETRIC',
    '--use-strided': 'ENET_USE_STRIDED',
    '--use-dsc': 'ENET_USE_DSC',
    '--separable-dilated': 'ENET_SEPARABLE_DILATED',
    '--context-pattern': 'ENET_CONTEXT_PATTERN',
    '--use-prelu': 'ENET_USE_PRELU',
    '--dsc-no-projection': 'ENET_DSC_NO_PROJECTION',
}


def nnunet_env():
    env = {'PYTHONHASHSEED': '42'}
    for name in ('raw', 'preprocessed', 'results'):
        env['nnUNet_' + name] = inside(REPO_ROOT / 'data' / ('nnUNet_' + name))
    return env


def run_folder(net_name):
    return RESULTS / (net_name + '__nnUNetPlans__2d') / 'fold_0'


def prepare_nnunet(env):
    # nnU-Net preprocessing and the project's fixed train/val split
    raw = REPO_ROOT / 'data' / 'nnUNet_raw' / DATASET_NAME
    prep = REPO_ROOT / 'data' / 'nnUNet_preprocessed' / DATASET_NAME
    if not (prep / 'nnUNetPlans.json').exists():
        sh(['nnUNetv2_plan_and_preprocess', '-d', str(DATASET_ID), '--verify_dataset_integrity'],
           container=TRAIN_CONTAINER, env=env)
    if (raw / 'splits_final.json').exists():
        shutil.copy(str(raw / 'splits_final.json'), str(prep / 'splits_final.json'))


def train(run):
    net_name = TRAINER_CLASS + '_' + run['config_name']
    out = run_folder(net_name)
    if (out / 'checkpoint_final.pth').exists():
        print(run['config_name'], 'is already trained, skipping')
        return
    env = nnunet_env()
    env.update(ARCH_ENV)
    env.update({'ENET_CHANNELS': model['channels'], 'ENET_EPOCHS': str(EPOCHS),
                'ENET_LR': run['lr'], 'ENET_OUTPUT_FOLDER': inside(out)})
    prepare_nnunet(env)
    cmd = ['nnUNetv2_train', DATASET_NAME, '2d', '0', '-tr', TRAINER_CLASS]
    if (out / 'checkpoint_latest.pth').exists():
        cmd.append('--c')           # resume the same schedule
    elif run['warm_start_from']:
        source = run_folder(TRAINER_CLASS + '_' + run['warm_start_from']) / 'checkpoint_best.pth'
        cmd += ['-pretrained_weights', inside(source)]
    sh(cmd, container=TRAIN_CONTAINER, env=env)


def collect(run):
    cmd = ['python3', 'nas/collect_results.py',
           '--net-name', TRAINER_CLASS + '_' + run['config_name'], '--stage', run['stage'],
           '--dataset-name', DATASET_NAME, '--dataset-id', str(DATASET_ID),
           '--channels', model['channels'], '--bottlenecks', ARCH_ENV['ENET_BOTTLENECKS'],
           '--trainer-class', TRAINER_CLASS]
    for flag, var in COLLECT_FLAGS.items():
        cmd += [flag, ARCH_ENV[var]]
    sh(cmd, container=TRAIN_CONTAINER, env=nnunet_env())


for run in model['runs']:
    if REUSE_FP32_CHECKPOINT:
        require(run_folder(TRAINER_CLASS + '_' + run['config_name']) / 'checkpoint_best.pth', 'REUSE_FP32_CHECKPOINT')
    else:
        train(run)
        collect(run)

### 3.6 Compare with the report

Reads each run's row from `nas/results.csv` and prints it next to the report's value. `nas/results.csv` already holds the rows of the runs behind the report, so this cell also works without re-training. A re-trained model will not match the report exactly, because GPU training is not bit-reproducible.

In [ ]:
def results_row(net_name):
    for row in read_csv(REPO_ROOT / 'nas' / 'results.csv'):
        if row['config_name'] == net_name:
            return row
    return None


for run in model['runs']:
    row = results_row(TRAINER_CLASS + '_' + run['config_name'])
    if row is None:
        print(run['config_name'], ': no row in nas/results.csv yet')
        continue
    print('{}: params {:,.0f} (report {:,}), FLOPs {:.1f} M (report {}), Dice {:.4f} (report {:.4f})'.format(
        run['config_name'], float(row['params']), model['params'],
        float(row['flops']) / 1e6, model['flops_m'], float(row['dice']), run['dice']))

**Status, Section 3.** The configuration values, parameter and FLOP counts and Dice values come from the report, the Slurm jobs and `nas/results.csv` (its rows for `12_dense_relu`, `12_dense_relu_warmstart150ep` and `12_dense_relu_u1` give Dice 0.7876, 0.7954 and 0.8279). The cells were not re-executed, and no training was re-run.

## 4. Cost-model calibration

The ILP prices every dataflow node (LUT, BRAM, DSP and cycles) with closed-form formulas of its bit widths and folding (P, Q). They start from FINN's own estimators and are calibrated by fitting constants to the post-route utilization of single nodes in earlier builds (Sec. IV-B6): ordinary least squares through the origin per node kind, on features such as PE × bits. The constants live in `ILP/finn_cost_model.py`, every fit is logged in `ILP/calibration.csv`, and the reasoning is in `ILP/finn_cost_model.md`. The calibration is specific to this architecture and its design choices (no URAM, no thresholding fused into the MVAU).

This section covers two things:

1. **Refit.** `dataflow/utils/fit_cost_model_from_node_csv.py` refits the constants from the node-level datasets (`dataflow/builds/*/results/node_resource_calibration_*.csv` and `dataflow/datasets/*.csv`). It prints every fit with its R², median error and leave-one-dataset-out error. It changes `ILP/finn_cost_model.py` only with `--apply`, which this notebook leaves off.
2. **Evaluation (Table V).** The error of the calibrated and of the uncalibrated (stock FINN estimators) model against the post-synthesis counts of the four constrained 150 FPS builds of Fig. 5 (INT4, INT6, INT8 and mixed 4/6/8), kept in `report/hw/constrained_builds/`. The builds are not re-run here: the evaluation reads their stored synthesis results.

Table V, absolute percentage error over the four builds:

| | LUT | BRAM | DSP |
|---|---|---|---|
| Uncalibrated, mean / max | 63.6% / 77.4% | 64.4% / 92.1% | 14.9% / 19.8% |
| Calibrated, mean / max | 13.6% / 25.6% | 5.9% / 7.9% | 9.4% / 12.0% |

With `REUSE_COST_MODEL = True` the constants in the repo and the stored synthesis results are used, and only the error summary is recomputed.

In [ ]:
CALIB_DIR = REPO_ROOT / 'report' / 'hw' / 'constrained_builds'
APPLY_REFIT = False     # True patches ILP/finn_cost_model.py with the refitted constants: leave False for a pure check

if not REUSE_COST_MODEL:
    refit = ['python3', 'dataflow/utils/fit_cost_model_from_node_csv.py']
    if APPLY_REFIT:
        refit.append('--apply')
    sh(refit, container=TRAIN_CONTAINER)
    # synthesis utilization -> stock-FINN estimate -> modelled_vs_real_cost.csv
    for script in ('build_synth_utilization_csv.py', 'build_uncalibrated_cost.py', 'plot_constrained_builds.py'):
        sh(['python3', 'report/hw/constrained_builds/' + script], container=TRAIN_CONTAINER)

# error of each model against the synthesis counts -> model_error_by_design.csv, model_error_summary.csv
sh(['python3', 'report/hw/constrained_builds/build_error_summary.py'], container=TRAIN_CONTAINER)

REPORT_TABLE_V = {   # (uncalibrated mean, uncalibrated max, calibrated mean, calibrated max), in %
    'LUT': (63.6, 77.4, 13.6, 25.6),
    'BRAM': (64.4, 92.1, 5.9, 7.9),
    'DSP': (14.9, 19.8, 9.4, 12.0),
}
DESIGNS = ['INT4', 'INT6', 'INT8', 'Mixed 4/6/8']
errors = {}
for row in read_csv(CALIB_DIR / 'model_error_by_design.csv'):
    resource = 'BRAM' if row['resource'] == 'BRAM18' else row['resource']
    errors[(resource, row['model'])] = [abs(float(row[d])) for d in DESIGNS]

print('absolute error in %, mean / max over the four builds (report in brackets)')
for resource, (u_mean, u_max, c_mean, c_max) in REPORT_TABLE_V.items():
    u = errors[(resource, 'uncalibrated')]
    c = errors[(resource, 'calibrated')]
    print('{:5s} uncalibrated {:5.1f} / {:5.1f} ({} / {})   calibrated {:5.1f} / {:5.1f} ({} / {})'.format(
        resource, sum(u) / len(u), max(u), u_mean, u_max, sum(c) / len(c), max(c), c_mean, c_max))

**Status, Section 4.** The stored `model_error_by_design.csv` gives the Table V values (checked by reading the file). The refit, the regeneration chain (`build_synth_utilization_csv.py`, `build_uncalibrated_cost.py`, `plot_constrained_builds.py`) and the cell itself were not run.

## 5. ILP solver, max-FPS mode

Sec. IV-B3, IV-B4 and IV-B7. In max-FPS mode the bit widths are fixed (here uniform 8 bit for weights and activations), so only the folds (P, Q) of every node are chosen, with two ILPs in sequence: the first minimizes the cycles per frame of the slowest node, the second minimizes the mean LUT, BRAM and DSP use while staying within a tolerance of that frame time. All caps are 100% of the XCZU7EV. The FIFO depths are then tuned with the block simulator.

- **Input:** the solver configuration `ILP/configs/config_12_dense_relu_warmstart150ep.py`, the Hessian sensitivity file, the calibrated cost model (Section 4)
- **Output:** per-layer bits (8) and folding, and the FIFO depths, as `layer_bits_folding_*.json`

The steps:

1. **Hessian sensitivity** (`ILP/layer_sensitivity.py`): Hutchinson trace estimate with 40 probes over all 1200 images (Sec. II-F). With the bits fixed at 8, the sensitivities do not influence the max-FPS result: `finn_milp.py` still needs the file, and its values are irrelevant at a single bit width.
2. **Solve** (`ILP/artifacts/S12_512_bilin_int8_maxfps/run_maxfps.sh`): pass A `--max-fps` finds the highest FPS the model allows; pass B `--target-fps <that FPS> --min-resources` finds the cheapest folding at exactly that FPS. Pass B is the folding that goes on to FIFO tuning.
3. **FIFO tuning** (`ILP/analytical/net_explicit.py`): simulates the given folding block by block, then in adjacent pairs, then as a whole network, with a paced source at the target rate, and sizes the FIFOs (Sec. IV-B7).

Stored result of pass A (`maxfps/summary.csv`): optimal, 8.0 average weight and activation bits, 51.6% of the device LUTs, 39.1% of the BRAM, 17.0% of the DSPs (293), and 305.2 FPS at the solver's 100 MHz model clock, limited by `initial.pool`. The deployed design runs at 187.48 MHz and was measured at 354.8 FPS on the board (Section 8).

In [ ]:
ILP_CONFIG = 'config_12_dense_relu_warmstart150ep'
SENSITIVITY = REPO_ROOT / 'ILP' / 'artifacts' / 'layer_sensitivity_12_dense_relu_warmstart150ep_n1200p40_seed0.json'
ILP_DIR = REPO_ROOT / 'ILP' / 'artifacts' / 'S12_512_bilin_int8_maxfps'

# 5.1 Hessian sensitivity (needs the FP32 checkpoint of Section 3 and a GPU)
if REUSE_HESSIAN:
    require(SENSITIVITY, 'REUSE_HESSIAN')
else:
    sh(['python3', 'ILP/layer_sensitivity.py', '--config', ILP_CONFIG, '--n-batches', '1200', '--n-probes', '40',
        '--seed', '0', '--out-file', inside(SENSITIVITY)], container=TRAIN_CONTAINER)

In [ ]:
# 5.2 Solve: pass A (max FPS) and pass B (cheapest folding at that FPS), as in run_maxfps.sh
if REUSE_ILP_SOLUTION:
    require(ILP_DIR / 'maxfps' / 'layer_bits_folding_maxfps.json', 'REUSE_ILP_SOLUTION')
    require(ILP_DIR / 'pinned_minres' / 'layer_bits_folding_pinned_minres.json', 'REUSE_ILP_SOLUTION')
else:
    # rewrites maxfps/ and pinned_minres/: back them up first (archive_pre_<YYYYMMDD>_<reason>/ convention)
    sh(['bash', 'ILP/artifacts/S12_512_bilin_int8_maxfps/run_maxfps.sh'], container=TRAIN_CONTAINER)

for name in ('maxfps', 'pinned_minres'):
    row = read_csv(ILP_DIR / name / 'summary.csv')[0]
    print('{:14s} {} | {} bit | LUT {:.1f}% BRAM {:.1f}% DSP {:.1f}% | {:.1f} FPS at {} MHz | bottleneck {}'.format(
        name, row['status'], row['avg_weight_bits'], float(row['lut_pct_of_budget']), float(row['bram_pct_of_budget']),
        float(row['dsp_pct_of_budget']), float(row['fps']), row['clock_mhz'], row['bottleneck_node']))

In [ ]:
# 5.3 FIFO tuning of pass B's folding -> simfifo_pairs/layer_bits_folding_S12_512_bilin_int8_maxfps_pairs.json
FIFO_TAG = 'S12_512_bilin_int8_maxfps_pairs'
FOLDING = ILP_DIR / 'simfifo_pairs' / ('layer_bits_folding_' + FIFO_TAG + '.json')

if REUSE_FIFO_TUNING:
    require(FOLDING, 'REUSE_FIFO_TUNING')
else:
    sh(['python3', 'ILP/analytical/net_explicit.py',
        inside(ILP_DIR / 'pinned_minres' / 'layer_bits_folding_pinned_minres.json'),
        '--out-dir', inside(ILP_DIR / 'simfifo_pairs'), '--tag', FIFO_TAG, '--workers', '6', '--max-iterations', '3'],
       container=TRAIN_CONTAINER)

**Status, Section 5.** The commands come from `run_maxfps.sh`, the `net_explicit.py` docstring and `layer_sensitivity.py`'s arguments, and the stored outputs listed above exist. The `layer_sensitivity.py` flags (1200 images, 40 probes, seed 0) are inferred from the stored file name `..._n1200p40_seed0.json`, and the `net_explicit.py` flags from its docstring and log: both are unverified. Nothing was re-run.

## 6. QAT

Sec. IV-B8: the quantizer scales are calibrated on the training images, then the network is fine-tuned with QAT in Brevitas for 15 epochs under the same training conditions as the FP32 training. QAT runs on `LayerQuantEnetFINN` (`enet/nnunetv2/nets/LayerQuantEnetFINN.py`), the network that is exported.

- **Input:** the FP32 checkpoint (Section 3) and a per-site bit file with every one of the 85 weight and 110 activation sites at 8 bit: `ILP/artifacts/archive/S12_dense_bilinear_512_uniform_v1/layer_bits_SITES_uniform_int8.json`. Folding is not part of QAT, accuracy depends only on the bits.
- **Output:** the INT8 checkpoint `nnUNetTrainerLayerQuantEnetFINN_12_dense_relu_bilinear_upsample_256_perlayer_12_dense_relu_warmstart150ep_bilinear_uniform_int8_perlayer_ft15ep`.
- **Steps:** (1) calibrate with `nas/post-quantization/calibrate_12_dense_relu_bilinear_upsample_256_perlayer.py --model-class finn` on all 1200 images (seed 0); (2) QAT with the trainer `nnUNetTrainerLayerQuantEnetFINN_12_dense_relu_bilinear_upsample_256_perlayer` for 15 epochs (seed 1234, batch norm not frozen); (3) `nas/collect_results.py` on `checkpoint_best.pth` only.

The report's QAT runs were Slurm jobs on Snellius, like the FP32 training (see 3.5). There is no stored job for this INT8 run. The cell adapts the steps of `nas/slurm/qat_12_dense_relu_bilinear_upsample_256_uniform_int6_ft15ep.job`, the closest stored job (INT6, on Dataset510, 2 h limit on one `gpu_mig` GPU).

**Dice.** The report gives 0.7946 after QAT (Sec. V-B2). The stored results give 0.7930 for the QAT model evaluated in software (`nas/results.csv`, `report/hw/final_results/final_results_s12_512_bilin_int8.csv`) and 0.7943 for the same network on the board (Section 8). A value of 0.7946 for the FPGA design was not found in the stored result files.

In [ ]:
QAT_TRAINER = 'nnUNetTrainerLayerQuantEnetFINN_12_dense_relu_bilinear_upsample_256_perlayer'
QAT_RUN = '12_dense_relu_warmstart150ep_bilinear_uniform_int8_perlayer_ft15ep'
QAT_NET = QAT_TRAINER + '_' + QAT_RUN
FP32_NET = TRAINER_CLASS + '_12_dense_relu_warmstart150ep'
CALIBRATED_NET = 'nnUNetTrainerLayerQuantEnetFINN_12_dense_relu_warmstart150ep_bilinear_uniform_int8_calibrated'
BITS_FILE = REPO_ROOT / 'ILP' / 'artifacts' / 'archive' / 'S12_dense_bilinear_512_uniform_v1' / 'layer_bits_SITES_uniform_int8.json'
QAT_CKPT = run_folder(QAT_NET) / 'checkpoint_best.pth'


def qat():
    env = nnunet_env()
    prepare_nnunet(env)
    # step 1: calibrate LayerQuantEnetFINN from the FP32 checkpoint (skipped if already done)
    if existing(run_folder(CALIBRATED_NET) / 'checkpoint_best.pth') is None:
        sh(['python3', 'nas/post-quantization/calibrate_12_dense_relu_bilinear_upsample_256_perlayer.py',
            '--model-class', 'finn', '--source-net-name', FP32_NET, '--out-net-name', CALIBRATED_NET,
            '--dataset-name', DATASET_NAME, '--layer-bits-file', inside(BITS_FILE),
            '--n-calibration-images', '1200', '--pad-to-multiple', '8', '--calibration-seed', '0'],
           container=TRAIN_CONTAINER, env=env)
    # step 2: 15 epochs of QAT (warm start = the calibrated checkpoint)
    out = run_folder(QAT_NET)
    env.update({'ENET_LAYER_BITS_FILE': inside(BITS_FILE),
                'ENET_PRETRAINED_CHECKPOINT': inside(run_folder(CALIBRATED_NET) / 'checkpoint_best.pth'),
                'ENET_EPOCHS': '15', 'ENET_FREEZE_BN': '0', 'ENET_CHECKPOINT_EVERY': '5', 'ENET_SEED': '1234',
                'ENET_OUTPUT_FOLDER': inside(out)})
    if not (out / 'checkpoint_final.pth').exists():
        cmd = ['nnUNetv2_train', DATASET_NAME, '2d', '0', '-tr', QAT_TRAINER]
        if (out / 'checkpoint_latest.pth').exists():
            cmd.append('--c')
        sh(cmd, container=TRAIN_CONTAINER, env=env)
    # step 3: evaluate checkpoint_best.pth only
    sh(['python3', 'nas/collect_results.py', '--net-name', QAT_NET, '--stage', QAT_RUN,
        '--dataset-name', DATASET_NAME, '--dataset-id', str(DATASET_ID),
        '--channels', '4,16,32,16,4', '--bottlenecks', '4,8,8,2,1', '--decoder-type', 'upsample_conv',
        '--use-asymmetric', '0', '--use-prelu', '0', '--context-pattern', 'dense_dilation',
        '--separable-dilated', '0', '--quant-bits', '32', '--trainer-class', QAT_TRAINER,
        '--checkpoint-name', 'checkpoint_best.pth'], container=TRAIN_CONTAINER, env=nnunet_env())


if REUSE_QAT_CHECKPOINT:
    require(QAT_CKPT, 'REUSE_QAT_CHECKPOINT')
else:
    qat()

row = results_row(QAT_NET)
if row:
    print('QAT model: Dice {:.4f} in nas/results.csv (report 0.7946, board 0.7943)'.format(float(row['dice'])))

**Status, Section 6.** The stored QAT checkpoint (`checkpoint_best.pth.txt` and the epoch 5, 10 and 15 checkpoints) and its `nas/results.csv` row (Dice 0.7930) exist. The QAT commands are adapted from the INT6 job and were not run; the bit file is read from `ILP/artifacts/archive/`, where it sits now (the export script's docstring still names an older path).

## 7. Deploy with FINN

Sec. IV-B8 and V-B2. The quantized checkpoint, the folding and the FIFO depths go into FINN, which makes the network hardware-compatible with custom graph transformations and generates the stitched Vivado IP, which is then built into the bitstream. The FINN container and the build outputs are on the build PC.

- **Input:** the INT8 checkpoint (Section 6) and the FIFO-tuned folding (Section 5)
- **Output:** stitched IP per partition, then the system design and bitstream (`.xsa`)
- **Expected result:** 157k LUT (68%), 45k LUTRAM, 143k FF, 156 BRAM (50%), 449 DSP (26%), no URAM, at 187.48 MHz (Sec. V-B2)

### 7.1 Eight partitions

The network is built as **8 separate FINN partitions**, not as one design. Each partition is synthesized and stitched into its own IP, and the partitions are joined afterwards in Vivado (step 5 below). The reason is build time. Stitching one monolithic graph hit a near-quadratic blow-up in Vivado's `create_bd_cell` (seen on the earlier S19 network: about 1,665 cells in one stitch), and even a 5-way split left one partition (stage 2/3, 419 nodes) that took more than two days. Eight smaller partitions keep the IP stitch phase and the out-of-context synthesis short, and they build in parallel.

The cuts are found from the graph structure, not from node indices (`dataflow/finn_stage_partition.py`):

| Partition | Content |
|---|---|
| 0 | initial block |
| 1 | stage 1: down1 and regular1 |
| 2 to 5 | stage 2/3 in four roughly equal quarters (partition 2 starts with down2's max-pool) |
| 6 | stage 4: up4 and regular4 |
| 7 | stage 5: up5, regular5 and the final layer |

**Multiple partitions are not smooth in FINN.** FINN can split a graph into partitions, but its flow is written around a single design. Every partition is built on its own, so FINN numbers its nodes from zero again in each one, and the same generic names (`MVAU_hls_0`, `StreamingFIFO_rtl_0`, ...) appear in all eight. FINN also bakes the node name into the name of the generated HLS and RTL module. Joined in one Vivado block design, those names collide. The build therefore overrides the names of all child IPs, with the partition name as prefix:

- **Node names.** Every node of a partition is renamed to `<partition name>_...` (`GiveUniqueNodeNames(prefix)`, with the partition name `GenericPartition_<i>`) before code generation (`step_hw_codegen`, `PrepareIP`, `HLSSynthIP`), because the name is fixed once the IP is generated. The folding config is matched on the bare names that FINN derives itself, so the rename has to come after folding and before code generation. Nodes that `SplitLargeFIFOs` adds later are renamed in a second pass. The code is in `_build_one_partition` of `dataflow/finn_s12_build.py`.
- **Stitched IP.** Each partition's stitched IP is named after the partition (`CreateStitchedIP`), so every partition has its own IP-XACT identity.
- **Merged RTL.** When the partitions' RTL is merged for the combined RTL simulation, every module declared inside a partition is renamed with the partition prefix as well (`_rename_partition_verilog_sources` in `dataflow/finn_partition_build_steps.py`). Verilator keeps only one definition per module name, which once wired one partition's ROM constants into another.
- **IODMA cores.** The input and the output DMA core of the 256 × 256 build hit the same collision: both compile to a Verilog module named `IODMA_hls_0`, with different ports. It was fixed with `dataflow/_rename_iodma_sources.py` and the `dataflow/fix_iodma_*.tcl` scripts.

### 7.2 FIFO sizes are overridden by graph surgery

FINN sizes FIFOs itself, by simulation, and does not support taking per-FIFO depths from an external plan. Here the depths come from the ILP and the block simulator (Section 5), so the build overrides FINN's FIFOs by editing the graph. The code is in `dataflow/finn_s12_build_steps.py`, called from `dataflow/finn_s12_build.py` for every partition:

1. **`step_set_fifo_depths_fixed2`** skips FINN's own FIFO autosizing (the slow RTL-simulation based one, which also produced very large depths on edges the ILP knew nothing about). It inserts a real StreamingFIFO node at depth 2 on every edge between hardware nodes, and keeps all of them as nodes so that none is optimized away.
2. **`step_force_fifo_depths_from_milp`** overwrites the depth of each of those FIFOs with the depth in the folding JSON (`intra_block_fifos` and `inter_block_fifos`).
   - Each FIFO is matched to the plan by the role of its producer and consumer node. The folding bridge (`build_partition_folding_config`) writes the plan per partition as `fifo_plan_partition<i>.json`, including a map from node name to role.
   - A partition's last FIFO has its consumer in the next partition, so it is matched by its producer alone.
   - The step refuses to run if a node name no longer points at the node the plan was made for. A renumbered graph once put a skip-branch depth on the main branch and deadlocked a partition.
   - A FIFO with no match keeps its stock depth (2). Every FIFO, matched or not, is listed in `fifo_force_report_partition_<i>.json`, and `flow_gates.gate_fifos_landed` checks that the forced depths landed.
3. **`SplitLargeFIFOs`** then splits any FIFO deeper than Vivado's `axis_data_fifo` limit (32768 words).

`finn_s12_build.py --fifo-autosize rtlsim` runs FINN's own sizer instead (the old behavior). The ILP's analytical FIFO model and the block simulator only replace it because FINN's sizing was far too large for this network (Sec. IV-B2).

### 7.3 Steps

Scripts are in `dataflow/builds/S12_dense_512_bilinear_int8_maxfps_v1/`:

1. **QONNX export** (training environment): `finn_export_S12_dense_512_bilinear_int8_maxfps_v1_ft15ep.py` writes `outputs/quantEnet_..._ft15ep_u8in.onnx` (uint8 input: z-score, quantize, +128), the conv-order JSON and a verification reference.
2. **Copy into the FINN container**: the ONNX, the conv-order JSON, the folding JSON and the helper scripts `dataflow/*.py`, flat into `/home/thelegendiv/finn/notebooks/enet/`.
3. **Preamble and 8-way build** (FINN container): `_launch_bilinear_int8_maxfps_preamble.sh` then `_launch_bilinear_int8_maxfps_full.sh`, which run `finn_s12_preamble.py` (streamline, convert to hardware, tag the 8 partitions) and `finn_s12_build.py --partitions all` (folding bridge, per-partition renaming, codegen and IP generation, the FIFO override of 7.2, stitching, out-of-context synthesis, RTL simulation). Both scripts start the job in the background with `nohup`. The full-build script carries the timestamped preamble directory of the original run (`PDIR`): edit it to the directory your preamble run created.
4. **The final design is assembled from three builds** (`results/README.md`): partitions 2, 3, 4 and 6 from the 8-way run; partitions 1, 5 and 7 from a rebuild with a name-stable FIFO fix (`_launch_p157_rebuild.sh`, `_launch_p7_labelselect_pe5.sh`); partition 0 with the initial pool lowered through InferPool (`_launch_ipool_preamble.sh`, `_launch_ipool_p0.sh`). `_launch_golden_int8.sh` runs the per-partition golden check of the assembled design.
5. **System design**: `vivado/run_bilin_int8_vivado_project.sh` copies a hand-edited block-design project (PS clocking, latency monitor, IODMA IPs) and points its partition IP repositories at the builds above. The hardware platform files are in `deployment/xsa/`; the design measured in the report is **`s12_512_bilin_int8_v2.xsa`**.

TODO: the post-implementation utilization report behind the expected numbers is among the build outputs on the build PC. Copy it into the repo, next to the `.xsa`.

In [ ]:
BUILD = REPO_ROOT / 'dataflow' / 'builds' / 'S12_dense_512_bilinear_int8_maxfps_v1'
ONNX = BUILD / 'outputs' / 'quantEnet_S12_dense_512_bilinear_int8_maxfps_v1_ft15ep_u8in.onnx'
CONV_ORDER = BUILD / 'outputs' / 'quantEnet_S12_dense_512_bilinear_int8_maxfps_v1_conv_order.json'

# 7.1 QONNX export of the QAT checkpoint (training environment)
if REUSE_QONNX_EXPORT:
    require(ONNX, 'REUSE_QONNX_EXPORT')
    require(CONV_ORDER, 'REUSE_QONNX_EXPORT')
else:
    sh(['python3', 'dataflow/builds/S12_dense_512_bilinear_int8_maxfps_v1/'
        'finn_export_S12_dense_512_bilinear_int8_maxfps_v1_ft15ep.py'], container=TRAIN_CONTAINER)

In [ ]:
# 7.2 - 7.5 FINN build. The container is on the build PC; with REUSE_FINN_BUILD = True the stored results are listed.
FINN_DIR = '/home/thelegendiv/finn/notebooks/enet'
REPORT_XSA = REPO_ROOT / 'deployment' / 'xsa' / 's12_512_bilin_int8_v2.xsa'     # the design measured in the report

if REUSE_FINN_BUILD:
    for d in (BUILD / 'results', REPO_ROOT / 'deployment' / 'xsa'):
        print(d.relative_to(REPO_ROOT).as_posix())
        for p in sorted(d.iterdir()):
            print('   ', p.name)
    require(REPORT_XSA, 'REUSE_FINN_BUILD')
else:
    inputs = [ONNX, CONV_ORDER, FOLDING] + sorted(p for p in (REPO_ROOT / 'dataflow').glob('*.py') if not p.name.startswith('_tmp'))
    for f in inputs:
        subprocess.run(['docker', 'cp', str(f), FINN_CONTAINER + ':' + FINN_DIR + '/'], check=True)
    scripts = ['_launch_bilinear_int8_maxfps_preamble.sh', '_launch_bilinear_int8_maxfps_full.sh']
    for s in scripts:
        subprocess.run(['docker', 'cp', str(BUILD / s), FINN_CONTAINER + ':' + FINN_DIR + '/'], check=True)
    print('inputs copied. Run these in order, waiting for each job to finish (they start in the background):')
    for s in scripts:
        print('  docker exec ' + FINN_CONTAINER + ' bash ' + FINN_DIR + '/' + s)
    print('Edit PDIR in the full-build script to the preamble directory the first job created.')
    print('Then the partition rebuilds and the Vivado project of steps 7.4 and 7.5 (scripts named above).')

**Status, Section 7.** Not verified: the FINN container and the build outputs are on the build PC, not on the machine this notebook was prepared on. The scripts and file names come from the repo (`_launch_*.sh`, `finn_export_*_ft15ep.py`, `results/README.md`). The stored QONNX export, the per-partition build results and the `.xsa` files are in the repo and are what the next section uses. The list of helper scripts copied into the container (`dataflow/*.py`) is a best guess from the flat-directory convention in `AGENTS.md`.

## 8. Verify with the bare-metal app

Sec. V-C and Fig. 6. The processing system runs a bare-metal application that only moves images and results between the PC and the accelerator through the input and output DMA engines. The accelerator in the programmable logic runs on its own, without scheduling by the PS.

`deployment/collect_results.py` drives the board from the PC over TCP (`lwip_collect_results` on the board, port 7, default address 192.168.0.111). For the 300 test images of Dataset509 it:

1. preprocesses each image into the board's uint8 input (z-score, the input quantizer scale from `deployment/img_intf/utils/input_profiles.json`, +128)
2. uploads all images into board DDR, runs the single-image latency pass and the timed back-to-back passes, with every time read from the PL tick counter (`axi_late_mon`, 187.481262 MHz)
3. downloads the class maps, scores Dice against `labelsTs`, and upserts a row into `deployment/results.csv`
4. with `--power`, samples the 18 INA226 rail monitors of the ZCU106 idle and under load and writes `deployment/power_results.csv`

- **Input:** the hardware platform `deployment/xsa/s12_512_bilin_int8_v2.xsa` (Section 7) programmed on the board, with the bare-metal application running
- **Expected result (report, Table III):** 354.8 FPS streaming throughput, 6.87 ms single-image latency, 16.1 mJ total and 5.2 mJ dynamic energy per image. The stored board row gives Dice 0.7943 (see the note in Section 6)

The sources of the bare-metal application (`lwip_collect_results`, `lwip_stream`) are on the build PC, not in this repo. `deployment/FINN_interface/` only holds Vitis IDE metadata.

TODO: copy the bare-metal sources from the build PC into the repo before handover (suggested place: `deployment/FINN_interface/`, where the Vitis workspace metadata already is)."

In [ ]:
BOARD_MODEL = 's12_512_bilinear_int8_maxfps'
BOARD_CONFIG = 'board_ipool'      # the design with the initial pool lowered through InferPool
BOARD_IP = '192.168.0.111'

if not REUSE_BOARD_RESULTS:
    # run on the PC connected to the board, with the bare-metal application running
    sh([sys.executable, 'deployment/collect_results.py', '--model-name', BOARD_MODEL, '--config', BOARD_CONFIG,
        '--ip', BOARD_IP, '--power'])

for row in read_csv(REPO_ROOT / 'deployment' / 'results.csv'):
    if row['model_name'] == BOARD_MODEL and row['config'] == BOARD_CONFIG:
        print('board: Dice {:.4f}, split_fps {}, hw_infer_fps {}, hw_latency_ms {}'.format(
            float(row['dice']), row.get('split_fps'), row.get('hw_infer_fps'), row.get('hw_latency_ms')))
for row in read_csv(REPO_ROOT / 'deployment' / 'power_results.csv'):
    if row['model_name'] == BOARD_MODEL and row['condition'].startswith('Inference'):
        print('power: {} W, {} mJ per image, {} mJ dynamic'.format(
            row['board_power_w'], row['energy_per_image_mj'], row['dynamic_energy_per_image_mj']))

**Status, Section 8.** Not run: no board here, and the bare-metal sources are on the build PC. The command and options come from `collect_results.py`'s own help. The stored board results are `deployment/results.csv` (Dice 0.7943) and `deployment/power_results.csv` (16.092 mJ total, 5.22 mJ dynamic, 354.76 FPS), which match the FPGA row of Table III.

## 9. Platform comparison

Sec. V-C and Table III. The same uniform 8-bit network is run on four platforms on the 300 test images of ARCADE, at batch size 1, with data loading and preprocessing excluded from the timing:

| Platform | Runtime | Device |
|---|---|---|
| GPU | TensorRT, INT8 | RTX 2050 laptop GPU |
| CPU | ONNX Runtime, int8 kernels | i7-1260P |
| FPGA, dataflow | FINN (this flow) | ZCU106 |
| FPGA, DPU | Vitis AI, two B4096 cores, 300 MHz | ZCU104 |

Latency is the time of one image from submission to result. Throughput is the sustained rate over a back-to-back stream of the 300 images. Energy per image is the energy over that stream divided by 300, total and above idle (dynamic). The power-measurement boundary differs per platform (see Section 10).

Report, Table III:

| Platform | Latency (ms) | Streaming FPS | Total energy (mJ/img) | Dynamic energy (mJ/img) |
|---|---|---|---|---|
| GPU (RTX 2050), TensorRT | 3.05 ± 0.19 | 281.7 ± 31.4 | 87.9 | 69.2 |
| CPU (i7-1260P), ONNX Runtime | 10.16 ± 0.92 | 97.2 ± 2.5 | 471.8 | 372.8 |
| FPGA, dataflow (ZCU106), FINN | 6.87 ± 0 | 354.8 ± 0 | 16.1 | 5.2 |
| FPGA, DPU (ZCU104), Vitis AI | 12.17 ± 0.01 | 120.13 ± 0.03 | 79.7 | 11.7 |

### 9.1 CPU and GPU

Run in the training environment, with the laptop plugged in and nothing else using the GPU or CPU (both throttle).

1. `report/hw/benchmark/export_int8_qcdq.py` converts the FINN QONNX export of Section 7 into QuantizeLinear/DequantizeLinear ONNX files for ONNX Runtime and TensorRT, and checks them against the QONNX reference.
2. TensorRT 10.3 is installed separately, outside the PyTorch install: `report/hw/benchmark/INSTALL_TRT.md`.
3. `benchmark_int8_cpu_gpu.py --backend ort-cpu` and `--backend trt-gpu` time the 300 images and score Dice. With `--power` they also measure energy: the GPU through the NVML board-energy counter, the CPU through the Windows Energy Meter (RAPL package), which needs `host_rapl_sampler.ps1` running on the Windows host. Each run upserts its rows into `report/hw/final_results/benchmark_s12_512.csv`.
4. `run_benchmarks_with_clocks.sh <out_dir>` runs the whole CPU and GPU set (FP32 and INT8) with clock logging.

`run_trt_int8.sh` calls the benchmark through the old path `report/MILP/benchmark/`, which no longer exists. The cell below sets the same environment and calls the script at its current path instead.

### 9.2 DPU

The Vitis AI flow of `dpu/README.md`, in WSL with the image `xilinx/vitis-ai-pytorch-cpu:ubuntu2004-3.0.0.106` and the ZCU104 board: `./run_flow.sh 12_dense_relu_warmstart150ep_fp32 all` (setup, stage, inspect, quantize, compile, deploy, bench). The QAT model of Table III is fine-tuned on Snellius as a Slurm job (`dpu/slurm/qat_12_dense_relu_warmstart150ep_dpu.job`, see 3.5), exported and compiled with `qat_export` and `qat_compile`, and measured with `python dpu/collect_results.py --qat`.

### 9.3 FPGA, dataflow

The row comes from Section 8.

In [ ]:
BENCH_DIR = 'report/hw/benchmark'
TRT = '/opt/trt_pkgs'
TRT_ENV = {'PYTHONPATH': TRT,
           'LD_LIBRARY_PATH': ':'.join([TRT + '/tensorrt_libs', TRT + '/nvidia/cuda_runtime/lib',
                                        TRT + '/nvidia/cublas/lib', TRT + '/nvidia/cuda_nvrtc/lib'])}

if not REUSE_BENCHMARKS:
    sh(['python3', BENCH_DIR + '/export_int8_qcdq.py'], container=TRAIN_CONTAINER)
    sh(['python3', BENCH_DIR + '/benchmark_int8_cpu_gpu.py', '--backend', 'ort-cpu', '--power'], container=TRAIN_CONTAINER)
    sh(['python3', BENCH_DIR + '/benchmark_int8_cpu_gpu.py', '--backend', 'trt-gpu', '--power'],
       container=TRAIN_CONTAINER, env=TRT_ENV)
    # DPU: run ./run_flow.sh and dpu/collect_results.py --qat as in 9.2 (WSL and the ZCU104 board)
else:
    require(REPO_ROOT / 'report' / 'hw' / 'final_results' / 'benchmark_s12_512.csv', 'REUSE_BENCHMARKS')

In [ ]:
# Table III from the stored results, next to the report's values
REPORT = {   # label: (numeric_mode, (latency ms, streaming FPS, total mJ, dynamic mJ))
    'GPU, TensorRT int8': ('int8_trt', (3.05, 281.7, 87.9, 69.2)),
    'CPU, ONNX Runtime int8': ('int8_ort', (10.16, 97.2, 471.8, 372.8)),
    'FPGA, FINN': ('int8', (6.87, 354.8, 16.1, 5.2)),
    'DPU, Vitis AI QAT': ('int8_dpu_qat', (12.17, 120.13, 79.7, 11.7)),
}
bench = read_csv(REPO_ROOT / 'report' / 'hw' / 'final_results' / 'benchmark_s12_512.csv')
board_power = [r for r in read_csv(REPO_ROOT / 'deployment' / 'power_results.csv')
               if r['model_name'] == BOARD_MODEL and r['condition'].startswith('Inference')]


def stored(mode_name):
    lat = fps = total = dyn = None
    for r in bench:
        if r['numeric_mode'] != mode_name:
            continue
        if r['mode'] == 'single_image':
            lat = float(r['latency_ms_mean'])
        elif r['mode'] == 'stream_warm':
            fps = float(r['throughput_fps'])
            if r['energy_j_per_image']:
                total = 1000 * float(r['energy_j_per_image'])
                dyn = 1000 * float(r['energy_dyn_j_per_image'])
    if mode_name == 'int8' and board_power:      # the FPGA energy is in deployment/power_results.csv
        total = float(board_power[0]['energy_per_image_mj'])
        dyn = float(board_power[0]['dynamic_energy_per_image_mj'])
    return (lat, fps, total, dyn)


def fmt(values):
    return ' '.join('{:>7}'.format('-' if v is None else round(v, 2)) for v in values)


print('{:24s} {:>31s}   {:>31s}'.format('', 'stored: lat FPS tot dyn', 'report: lat FPS tot dyn'))
for label, (mode_name, values) in REPORT.items():
    print('{:24s} {}   {}'.format(label, fmt(stored(mode_name)), fmt(values)))

**Status, Section 9.** Nothing was run. In the stored results, the FPGA row (6.87 ms, 354.8 FPS, 16.1 and 5.2 mJ) and the DPU QAT row (12.17 ms, 120.13 FPS, 79.7 and 11.7 mJ) match Table III. The stored CPU and GPU rows differ from it (for example 44.7 FPS for the CPU against the report's 97.2, and 306.0 FPS for the GPU against 281.7). CPU and GPU results vary from run to run: they are non-deterministic and depend on the thermal state and clock behaviour of the laptop. `benchmark_s12_512.csv` holds the latest run (2026-11-08, `run_benchmarks_with_clocks.sh`), which is not expected to equal Table III, and a re-run will not reproduce it exactly either.

## 10. Limitations

### 10.1 Of the method (from the report)

- **One architecture, one task, one device.** The optimizer is demonstrated on one CNN, one segmentation task and one FPGA platform. Its broader applicability is not validated (Sec. VI).
- **The cost model is calibrated on this architecture** and its design choices, and may not generalize (Sec. IV-B6). After calibration the LUT and DSP counts are still underestimated, and the LUT error grows with the average bit width: mean absolute error 13.6% (max 25.6%) for LUT, 5.9% (7.9%) for BRAM, 9.4% (12.0%) for DSP (Table V). Resource caps therefore need a safety margin (Sec. V-B1).
- **The FIFO tuning simulator** is neither cycle-accurate nor hardware-faithful. Its result is validated afterwards by RTL simulation and by measurement on the board (Sec. IV-B7).
- **Design choices that restrict the flow** (Sec. IV-B2): URAM is not used, thresholding is not fused into the MVAU, PReLU is not implemented, and post-training pruning is not considered.
- **The QAT recipe is untuned:** 15 epochs under the FP32 training conditions (Sec. IV-B8).
- **Hessian sensitivity** assumes additive layer scores, which does not hold in practice, especially for activations (Sec. II-F). It is not used by the max-FPS flow of this notebook.

### 10.2 Of the measurements

- **The power-measurement boundaries differ across platforms** (Sec. V-C). The GPU figure covers the GPU board only, the CPU figure the processor package without memory, and the FPGA and DPU figures are sums of on-board rail monitors measured on two different boards (ZCU106 and ZCU104). The dynamic energy is the closer comparison. The report did not isolate why the FPGA uses less energy.
- **Throughput is measured at batch size 1.** Batching could raise CPU and GPU throughput at the cost of latency, and was not evaluated (Sec. V-C).
- **Dice differs slightly between platforms** (0.7901 to 0.7946) because each software stack needs its own conversion and fine-tuning steps (Sec. V-C).
- **CPU and GPU results vary from run to run.** They are non-deterministic and depend on the thermal state and clocks of the laptop, so a re-run gives different numbers from Table III. The dataflow accelerator's latency is deterministic (spread of 0 ms in the report).

### 10.3 Of this repo and handover

- **FINN is v0.10.1 with a cherry-pick.** The build uses FINN v0.10.1, not a newer release, plus a commit cherry-picked from the [`v1.0.0-alpha`](https://github.com/Xilinx/finn/tree/v1.0.0-alpha) branch that fixes a bug in the MVAU. The container reports the version `v0.10.1-10-g39f0c9a6b`, and its FINN tree was marked dirty (uncommitted changes). The cherry-picked commit is not recorded in this repo. The local clone `sources/FINN` is at the same base commit and does not contain it: the 10 commits after v0.10.1 there are dependency updates. A clean v0.10.1 or v1.0.0-alpha build will therefore not match the one in the report. TODO: record the commit hash, and any other uncommitted change in the container's FINN tree, or ship them as a patch file.
- **The network is built as 8 separate FINN partitions** and joined in Vivado afterwards, because a single stitch of the whole graph takes extremely long (Section 7.1). FINN's whole-network flow is therefore not used, and multiple partitions are not a smooth path in FINN: the names of all child IPs have to be overridden with the partition name as prefix, or they collide in the Vivado block design (Section 7.1). The cuts are found from the graph structure (max-pool and upsample nodes) and are specific to this ENet-style stage layout.
- **FIFO sizes are overridden by graph surgery**, which FINN does not support (Section 7.2). FINN's own FIFO sizing is bypassed, so FINN's FIFO estimates and its simulation-based depth checks do not apply to the built design. The depths rest on the ILP model, the block simulator, the per-partition RTL simulation and the measurement on the board. The override matches FIFOs to the plan by node name and role, and stops with an error if the graph no longer matches. A FIFO with no match keeps depth 2.
- **The FINN container, its build outputs and the bare-metal application sources are on the build PC**, not in this repo. The FINN build, the Vivado steps and the board stage were not re-run for this handover: Sections 7 and 8 reuse the stored results. The sources still have to be copied into the repo (Section 8).
- **The final design is assembled from three partial builds** (Section 7.3, step 4), and its block design was made by editing an earlier Vivado project by hand (`vivado/make_project_from_base.tcl`). That base project is in the FINN container on the build PC, not in this repo, so the system design cannot be regenerated from the repo alone. The `.xsa` files in `deployment/xsa/` are the shipped result; the report's design is `s12_512_bilin_int8_v2.xsa`.
- **Ours-L stops after training:** there is no ILP configuration, QAT bit file or build for it.
- **The stored Dice after QAT differs from the report:** the report gives 0.7946, the stored results 0.7930 in software and 0.7943 on the board (Section 6).
- **The Kaggle download was not checked** against the folder layout the builder script reads (Section 2).
- **Some paths in scripts and docstrings are stale:** `report/MILP/...` (now `report/hw/...`, including in `run_trt_int8.sh`), and `ILP/artifacts/S12_dense_bilinear_512_uniform_v1/` (now under `ILP/artifacts/archive/`).

## Appendix A. Max-accuracy flow (placeholder, off by default)

Not part of the max-FPS flow above. This placeholder marks where the report's maximum-accuracy flow (Sec. IV-B5, Fig. 5) goes. It is switched off by `RUN_MAX_ACCURACY_FLOW = False` in the Setup cell: with it off, the flow cell below does nothing.

**Setting.** Target 150 FPS at 100 MHz, resource caps of 40% of the LUTs, BRAM and DSPs of the XCZU7EV, weight and activation bit widths chosen per layer from 4, 6 and 8 bit, and the folding chosen at the same time. Two ILPs in sequence: the first minimizes the mean normalized Hessian sensitivity (the accuracy proxy), the second minimizes the mean resource use while keeping the sensitivity within a tolerance of 0.1% (`--max-accuracy-eps 0.001`) of the first. The result is compared with uniform 4, 6 and 8 bit builds, which fix the bits and only choose the folding.

**Steps**, as in Sections 5 to 7 with different solver arguments:

1. **Hessian sensitivity:** the same file as Section 5.1. It already holds the 4, 6 and 8 bit values.
2. **Solve** with `ILP/finn_milp.py` and the arguments in the cell below, taken from the stored `run_args.json`.
3. **Per-site bits and FIFO tuning:** `ILP/expand_layer_bits.py` expands the per-layer bits to the per-site bits that QAT and the export use, then `ILP/analytical/net_explicit.py` tunes the FIFOs.
4. **QAT** with those bits: `nas/slurm/qat_12_dense_relu_warmstart150ep_bilinear_ablation_v2_cap40_fps150_mixed468_ft15ep.job`. The three-seed runs are in `nas/slurm/qat_12_dense_relu_warmstart150ep_bilinear_int6_mixed468_seeds_ft15ep_array.job`.
5. **FINN build** of the folding, with the scripts of Section 7.

**Stored artifacts:**

- solver outputs: `ILP/artifacts/S12_512_ablation_v2/cap40_fps150/` with `mixed_468`, `uniform_int4` and `uniform_int6`. Uniform 8 bit is infeasible at these caps, so its solve is `ILP/artifacts/S12_512_ablation_v2/relaxed_fps150/uniform_int8` (caps 100%)
- FINN builds: `dataflow/builds/S12_512_ablation_v2/` (see its `README.md`), with launch scripts in `dataflow/builds/S12_dense_512_bilinear_*_cap40_v1/`
- results: `report/hw/constrained_builds/` (`dice_scores.csv`, `synth_utilization.csv`, `modelled_vs_real_cost.csv`)

**Expected result (Sec. V-B1, Fig. 5):**

| Build | Dice | LUT | BRAM | DSP |
|---|---|---|---|---|
| Uniform 6 bit | 0.7712 | 39% | 31% | 8.5% |
| Mixed 4/6/8 bit (average 7.86 weight, 6.91 activation bits) | 0.7843 | 40% | 40% | 15% |
| Uniform 8 bit | infeasible at the 40% caps (63% of the LUTs) | | | |

Shares are the estimated use of the XCZU7EV. Over three QAT seeds the mixed build reaches 0.7869 ± 0.0043 and the uniform 6-bit build 0.7736 ± 0.0020.

In [ ]:
MAXACC_OUT = REPO_ROOT / 'ILP' / 'artifacts' / 'S12_512_maxacc_rerun' / 'mixed_468'    # new folder: the stored artifacts stay untouched

if not RUN_MAX_ACCURACY_FLOW:
    print('max-accuracy flow is off (RUN_MAX_ACCURACY_FLOW = False in the Setup cell)')
else:
    # A.1 Hessian sensitivity: the file of Section 5.1, which already covers 4, 6 and 8 bit
    require(SENSITIVITY, 'REUSE_HESSIAN')

    # A.2 solve, with the arguments of the stored run_args.json (ILP/artifacts/S12_512_ablation_v2/cap40_fps150/mixed_468)
    MAXACC_OUT.mkdir(parents=True, exist_ok=True)
    sh(['python3', 'ILP/finn_milp.py', '--config', ILP_CONFIG, '--sensitivity-file', inside(SENSITIVITY),
        '--candidate-bits', '4,6,8', '--force-dsp',
        '--max-lut-fraction', '0.4', '--max-bram-fraction', '0.4', '--max-dsp-fraction', '0.4', '--max-uram-fraction', '0.4',
        '--target-fps', '150', '--clock-mhz', '100', '--time-limit', '300', '--gap-rel', '0.001',
        '--out-file', inside(MAXACC_OUT / 'layer_bits_folding_mixed_468.json')], container=TRAIN_CONTAINER)

    # A.3 TODO: per-layer bits to per-site bits (ILP/expand_layer_bits.py), then FIFO tuning (ILP/analytical/net_explicit.py)
    # A.4 TODO: QAT with the per-site bits (nas/slurm/qat_12_dense_relu_warmstart150ep_bilinear_ablation_v2_cap40_fps150_mixed468_ft15ep.job)
    # A.5 TODO: FINN build of the folding (dataflow/builds/S12_512_ablation_v2/cap40_fps150/mixed_468), as in Section 7
    raise NotImplementedError('steps A.3 to A.5 of the max-accuracy flow are not written yet')

In [ ]:
# Stored Dice of the constrained builds next to the report's values. Reads a CSV only, so it runs whatever the toggle says.
REPORT_MAXACC = {'fp32': 0.7954, 'int6': 0.7712, 'mixed468': 0.7843}    # FP32: Sec. V-B2; INT6 and mixed: Sec. V-B1
for row in read_csv(REPO_ROOT / 'report' / 'hw' / 'constrained_builds' / 'dice_scores.csv'):
    ref = REPORT_MAXACC.get(row['scheme'])
    note = '' if ref is None else ' (report {:.4f})'.format(ref)
    print('{:9s} {:13s} Dice {:.4f}{}'.format(row['scheme'], row['bits'], float(row['dice']), note))

**Status, Appendix A.** A placeholder. The solver arguments come from the stored `run_args.json` of the `mixed_468` solve. Steps A.3 to A.5 are not written, and nothing was run. The Dice cell above reads `dice_scores.csv`, which holds 0.7954 (FP32), 0.7712 (INT6) and 0.7843 (mixed 4/6/8), the same as the report.